# MyTravels — Preview Runbook

Runs the full MyTravels stack via Docker Compose. No local SDK or build tools required.

| Service | Purpose | Port(s) |
|---------|---------|--------|
| PostgreSQL | Primary database | 5432 |
| RabbitMQ | Message broker | 5672 · 15672 (UI) |
| MinIO | Object storage | 9000 (API) · 9090 (Console) |
| API | REST API | 5101 |
| Messaging | Background worker | 5102 |

> Run each cell in order.

## Summary

This runbook starts the complete MyTravels stack locally using Docker Compose — no local SDKs or build tools needed. It walks through:

1. **(Optional) Build & push images** — build all service images and push them to the container registry via `docker-compose.build.yml`.
2. **Copy environment file** — copy `.env.example` to `.env` if one doesn't already exist.
3. **Load environment variables** — read the `.env` file in this directory into the notebook's environment.
4. **Start the stack** — bring up PostgreSQL, RabbitMQ, MinIO, the API (port 5101), and the Messaging worker (port 5102) with `docker compose up -d`.
5. **Check service health** — verify all containers are running with `docker compose ps`.
6. **View logs** — tail recent logs for each service, including the DB migration jobs.
7. **Tear down** — stop everything and wipe volumes with `docker compose down --volumes`.

**Prerequisites:** Rancher Desktop (running), JupyterLab, and a `.env` file in this directory.

## Prerequisites

Rancher Desktop and JupyterLab install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

- A `.env` file must exist in this directory (copy from `.env.example` if one exists)

---
## 1. Copy environment file

Copy `.env.example` to `.env` if `.env` doesn't already exist.

In [ ]:
%%bash
if [ -f .env ]; then
  echo ".env already exists, skipping."
else
  cp .env.example .env
  echo "Copied .env.example to .env"
fi

---
## Build and Push Docker Images

Build all service images and push them to the container registry. This uses the `docker-compose.build.yml` file which defines the build targets and registry destination for each service.

### Using your own Docker Hub registry

The images are published under the `tshepontlhokoa` Docker Hub namespace. To push to your own registry:

1. Create a free account at [hub.docker.com](https://hub.docker.com) if you don't have one — your username becomes your registry namespace. (Repositories are created automatically on first push, so no need to create them manually.)
2. Find and replace `tshepontlhokoa/` with `<your-dockerhub-username>/` in **both** files:
   - `docker-compose.build.yml` — where the images are built and pushed
   - `docker-compose.yml` — where the same images are pulled when the stack starts
3. Log in before pushing: `docker login`

The three images affected are `mytravels-migrations`, `mytravels-api`, and `mytravels-messaging`.

> **Before running:** ensure you are logged in to your container registry (`docker login`) and that `docker-compose.build.yml` is configured with the correct image names and registry.

In [ ]:
%%bash
docker compose -f docker-compose.build.yml build --push

---
## 2. Load environment variables

In [ ]:
from pathlib import Path
import os

for line in Path(".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, _, value = line.partition("=")
        os.environ[key.strip()] = value.strip()

print("Loaded environment from .env")

---
## 3. Start the stack

In [ ]:
%%bash
docker compose up -d
echo "Stack started."

---
## 4. Check service health

In [ ]:
%%bash
docker compose ps

**Service UIs:**
- RabbitMQ Management: http://localhost:15672
- MinIO Console: http://localhost:9090
- API: http://localhost:5101
- Messaging: http://localhost:5102

---
## 5. View logs (optional)

In [ ]:
%%bash
echo "=== minio ===" && docker compose logs --tail=20 minio
echo "=== rabbitmq ===" && docker compose logs --tail=20 rabbitmq
echo "=== postgres ===" && docker compose logs --tail=20 postgres
echo "=== cleanup-migrations ===" && docker compose logs --tail=20 cleanup-migrations
echo "=== migrate-core-db ===" && docker compose logs --tail=20 migrate-core-db
echo "=== api ===" && docker compose logs --tail=20 api
echo "=== messaging ===" && docker compose logs --tail=20 messaging

In [ ]:
%%bash
# Stop and remove containers AND volumes — wipes all data
docker compose down --volumes
echo "Containers and volumes removed."